# 02 — Treino dos modelos (do zero)

Treina três CNNs **sem pesos pré-treinados** no PlantVillage (128×128) e avalia cada uma no teste do
PlantVillage e no teste do **PlantDoc** (fotos de campo).

| Run | Modelo |
|---|---|
| `cnn_base` | CNN própria (4 blocos conv) |
| `mnv3s` | MobileNetV3-Small |
| `mnv2_050` | MobileNetV2 α=0.5 |

**Como usar**
1. Ambiente de execução → GPU (L4 ou A100).
2. Rode primeiro com `QUICK = True` (~8 min): treina 2 épocas em 10% dos dados só para pegar erros.
3. Se tudo rodar, mude para `QUICK = False` e rode de novo a partir da célula de configuração.

**Se o Colab cair:** rode tudo de novo. Runs já concluídos são pulados e o run interrompido
continua da última época salva (`BackupAndRestore`).

**Saídas** (em `MyDrive/leafscan/runs/<run>/`): `best.keras`, `history.csv`, `metrics.json`,
`pred_pv_test.npy`, `pred_pd_test.npy`. Resumo em `MyDrive/leafscan/results/metrics_02.csv`.

## 1. Configuração

In [ ]:
QUICK = True   # True = teste rápido (10% dos dados, 2 épocas); False = treino completo

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, shutil
import numpy as np
import tensorflow as tf
import keras
from keras import layers

DRIVE_DIR = '/content/drive/MyDrive/leafscan'
CACHE_DIR = f'{DRIVE_DIR}/cache'
RUNS_DIR  = f'{DRIVE_DIR}/runs_quick' if QUICK else f'{DRIVE_DIR}/runs'
RES_DIR   = f'{DRIVE_DIR}/results'
LOCAL     = '/content/cache'

SEED       = 42
EPOCHS     = 2 if QUICK else 20
BATCH      = 128
LR         = 2e-3
WD         = 1e-4
PATIENCE   = 6
RUNS       = ['cnn_base', 'mnv3s', 'mnv2_050']

for d in (RUNS_DIR, RES_DIR, LOCAL):
    os.makedirs(d, exist_ok=True)

keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_global_policy('mixed_float16')
print('TF', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Carregar o cache (cópia local do Drive para leitura rápida)

In [ ]:
t0 = time.time()
for name in ('pv_x', 'pv_y', 'pv_split', 'pd_x', 'pd_y', 'pd_split'):
    dst = f'{LOCAL}/{name}.npy'
    if not os.path.exists(dst):
        shutil.copy(f'{CACHE_DIR}/{name}.npy', dst)
with open(f'{CACHE_DIR}/meta.json') as f:
    meta = json.load(f)
print(f'Cópia: {time.time()-t0:.0f}s')

PV_CLASSES = meta['pv_classes']
NUM_CLASSES = len(PV_CLASSES)
PD_IDX = np.array(meta['pd_pv_indices'])
IMG = meta['img_size']

pv_x = np.load(f'{LOCAL}/pv_x.npy'); pv_y = np.load(f'{LOCAL}/pv_y.npy').astype(np.int32)
pv_split = np.load(f'{LOCAL}/pv_split.npy')
pd_x = np.load(f'{LOCAL}/pd_x.npy'); pd_y = np.load(f'{LOCAL}/pd_y.npy').astype(np.int32)
pd_split = np.load(f'{LOCAL}/pd_split.npy')

x_tr, y_tr = pv_x[pv_split == 0], pv_y[pv_split == 0]
x_va, y_va = pv_x[pv_split == 1], pv_y[pv_split == 1]
x_te, y_te = pv_x[pv_split == 2], pv_y[pv_split == 2]
x_pd, y_pd = pd_x[pd_split == 2], pd_y[pd_split == 2]
del pv_x

if QUICK:
    rng = np.random.default_rng(SEED)
    i = rng.choice(len(y_tr), len(y_tr) // 10, replace=False); x_tr, y_tr = x_tr[i], y_tr[i]
    i = rng.choice(len(y_va), len(y_va) // 10, replace=False); x_va, y_va = x_va[i], y_va[i]

print('treino', x_tr.shape, '| val', x_va.shape, '| teste PV', x_te.shape, '| teste PlantDoc', x_pd.shape)

## 3. Pipeline de dados e aumento de dados (leve)
Aumento aplicado por imagem: as 8 simetrias do quadrado (flips + transposição), recorte aleatório
de 80–100% da área, brilho e contraste. É o aumento "padrão"; o aumento forte fica para o notebook 04.

In [ ]:
def augment_light(x, y):
    # x: lote float32 [B, H, W, 3] em 0..255
    b = tf.shape(x)[0]
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    t = tf.random.uniform([b, 1, 1, 1]) < 0.5
    x = tf.where(t, tf.transpose(x, [0, 2, 1, 3]), x)
    # recorte aleatório (80–100% do lado) + redimensionamento
    s = tf.random.uniform([b], 0.8, 1.0)
    oy = tf.random.uniform([b]) * (1 - s); ox = tf.random.uniform([b]) * (1 - s)
    boxes = tf.stack([oy, ox, oy + s, ox + s], axis=1)
    x = tf.image.crop_and_resize(x, boxes, tf.range(b), [IMG, IMG])
    # brilho e contraste
    x = x + tf.random.uniform([b, 1, 1, 1], -25.0, 25.0)
    m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - m) * tf.random.uniform([b, 1, 1, 1], 0.8, 1.2) + m
    return tf.clip_by_value(x, 0.0, 255.0), y

def make_ds(x, y, training, batch=BATCH, aug_fn=augment_light):
    with tf.device('/CPU:0'):
        X = tf.constant(x); Y = tf.constant(y)
    n = len(y)
    ds = tf.data.Dataset.range(n)
    if training:
        ds = ds.shuffle(n, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch)
    ds = ds.map(lambda i: (tf.cast(tf.gather(X, i), tf.float32), tf.gather(Y, i)),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training and aug_fn is not None:
        ds = ds.map(aug_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

ds_tr = make_ds(x_tr, y_tr, training=True)
ds_va = make_ds(x_va, y_va, training=False, batch=256)
steps_per_epoch = math.ceil(len(y_tr) / BATCH)

# conferência visual rápida do aumento
import matplotlib.pyplot as plt
xb, yb = next(iter(ds_tr))
fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
for k in range(8):
    ax[k].imshow(xb[k].numpy().astype('uint8')); ax[k].axis('off')
    ax[k].set_title(PV_CLASSES[int(yb[k])].split('___')[-1][:18], fontsize=7)
plt.show()

## 4. Modelos
Todos recebem pixels em 0..255 (o próprio modelo normaliza), assim o app só precisa enviar a imagem RGB.
A camada de saída fica em float32 por causa da precisão mista.

In [ ]:
def conv_bn(x, f):
    x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def fix_bn_momentum(base, momentum=0.99):
    # As MobileNets do Keras usam momentum 0.999 no BatchNorm (pensado para pesos já treinados).
    # Treinando do zero, as médias móveis demoram muito a se ajustar e a validação fica travada.
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.momentum = momentum

def build_model(name):
    inp = keras.Input((IMG, IMG, 3), name='image')
    if name == 'cnn_base':
        x = layers.Rescaling(1 / 255.0)(inp)
        for f in (32, 64, 128, 256):
            x = conv_bn(x, f); x = conv_bn(x, f)
            x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.3)(x)
    elif name == 'mnv3s':
        base = keras.applications.MobileNetV3Small(
            input_shape=(IMG, IMG, 3), include_top=False, weights=None,
            pooling='avg', include_preprocessing=True)   # escala 0..255 -> -1..1 internamente
        fix_bn_momentum(base)
        x = base(inp)
        x = layers.Dropout(0.2)(x)
    elif name == 'mnv2_050':
        x = layers.Rescaling(1 / 127.5, offset=-1.0)(inp)
        base = keras.applications.MobileNetV2(
            input_shape=(IMG, IMG, 3), alpha=0.5, include_top=False, weights=None, pooling='avg')
        fix_bn_momentum(base)
        x = base(x)
        x = layers.Dropout(0.2)(x)
    else:
        raise ValueError(name)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='probs')(x)
    return keras.Model(inp, out, name=name)

for r in RUNS:
    m = build_model(r)
    print(f'{r:10s} {m.count_params():>10,d} parâmetros')
    del m
keras.backend.clear_session()

## 5. Treino e avaliação

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

class EpochTime(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs['epoch_time_s'] = time.time() - self.t0

def predict_probs(model, x):
    return model.predict(make_ds(x, np.zeros(len(x), np.int32), training=False, batch=256), verbose=0)

def evaluate(model):
    p_pv = predict_probs(model, x_te)
    p_pd = predict_probs(model, x_pd)
    pred_pv = p_pv.argmax(1)
    pred_pd_38 = p_pd.argmax(1)                      # protocolo A: escolhe entre as 38 classes
    pred_pd_r = PD_IDX[p_pd[:, PD_IDX].argmax(1)]    # protocolo B: só classes presentes no PlantDoc
    m = {
        'pv_test_acc': accuracy_score(y_te, pred_pv),
        'pv_test_f1': f1_score(y_te, pred_pv, average='macro'),
        'pd_acc_38': accuracy_score(y_pd, pred_pd_38),
        'pd_f1_38': f1_score(y_pd, pred_pd_38, labels=PD_IDX, average='macro', zero_division=0),
        'pd_acc_restrito': accuracy_score(y_pd, pred_pd_r),
        'pd_f1_restrito': f1_score(y_pd, pred_pd_r, labels=PD_IDX, average='macro', zero_division=0),
    }
    return {k: float(v) for k, v in m.items()}, p_pv, p_pd

def run(name):
    rdir = f'{RUNS_DIR}/{name}'
    os.makedirs(rdir, exist_ok=True)
    if os.path.exists(f'{rdir}/metrics.json'):
        print(f'[{name}] já concluído, pulando')
        return json.load(open(f'{rdir}/metrics.json'))

    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_model(name)
    sched = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0, decay_steps=EPOCHS * steps_per_epoch,
        warmup_target=LR, warmup_steps=steps_per_epoch)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=sched, weight_decay=WD),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    t0 = time.time()
    if not os.path.exists(f'{rdir}/train_done'):
        model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS, verbose=2, callbacks=[
            keras.callbacks.BackupAndRestore(f'{rdir}/backup'),
            EpochTime(),
            keras.callbacks.ModelCheckpoint(f'{rdir}/best.keras', monitor='val_accuracy', save_best_only=True),
            keras.callbacks.CSVLogger(f'{rdir}/history.csv', append=True),
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE),
        ])
        open(f'{rdir}/train_done', 'w').close()
    train_time = time.time() - t0

    best = keras.models.load_model(f'{rdir}/best.keras', compile=False)
    metrics, p_pv, p_pd = evaluate(best)
    hist = pd.read_csv(f'{rdir}/history.csv')
    metrics.update({
        'run': name,
        'params': int(best.count_params()),
        'epochs': int(len(hist)),
        'best_val_acc': float(hist['val_accuracy'].max()),
        'train_time_min': float(hist['epoch_time_s'].sum() / 60) if 'epoch_time_s' in hist else train_time / 60,
    })
    np.save(f'{rdir}/pred_pv_test.npy', p_pv.astype(np.float16))
    np.save(f'{rdir}/pred_pd_test.npy', p_pd.astype(np.float16))
    json.dump(metrics, open(f'{rdir}/metrics.json', 'w'), indent=2)
    print(f'[{name}] PV acc={metrics["pv_test_acc"]:.4f} | PlantDoc acc(38)={metrics["pd_acc_38"]:.4f} '
          f'| PlantDoc acc(restrito)={metrics["pd_acc_restrito"]:.4f} | {metrics["train_time_min"]:.1f} min')
    return metrics

results = [run(r) for r in RUNS]

## 6. Resumo

In [ ]:
df = pd.DataFrame(results).set_index('run')
cols = ['params', 'epochs', 'train_time_min', 'best_val_acc', 'pv_test_acc', 'pv_test_f1',
        'pd_acc_38', 'pd_f1_38', 'pd_acc_restrito', 'pd_f1_restrito']
df = df[cols]
if not QUICK:
    df.to_csv(f'{RES_DIR}/metrics_02.csv')
    print('Salvo em', f'{RES_DIR}/metrics_02.csv')
df.style.format({c: '{:.4f}' for c in cols[3:]} | {'train_time_min': '{:.1f}', 'params': '{:,}'})